# مصنع تدريب السرب الذكي (10 محولات LoRA لمرة واحدة على Colab T4)
- النموذج الأساسي الصغير: `Qwen/Qwen2.5-0.5B-Instruct` (~500MB، يعمل في 3GB RAM على GitHub Actions)
- المخرجات: 10 ملفات محولات LoRA صغيرة (~4–18MB لكل وكيل) محفوظة داخل `lora_adapters/` في GitHub بدون أي اعتماد على OpenAI أو خوادم مدفوعة.


In [ ]:
%cd /content
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/sdxdxa56-design/jdjsjsh.git"
if not Path("/content/zh_swarm10/.git").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/zh_swarm10"], check=True)
else:
    subprocess.run(["git", "-C", "/content/zh_swarm10", "pull", "--ff-only"], check=False)
%cd /content/zh_swarm10
!pip install -q --upgrade transformers peft accelerate datasets safetensors bitsandbytes


In [ ]:
# تدريب الـ 10 وكلاء (LoRA Adapters r=8) على بطاقة T4 المجانية
!python3 -m swarm10.colab_train_10_lora --epochs 3 --rank 8 --output-dir lora_adapters


In [ ]:
# التحقق من الأحجام (أقل من 50MB لكل محول) ودفع المجلد إلى GitHub
import getpass
from pathlib import Path
import subprocess

for p in sorted(Path("lora_adapters").rglob("*.safetensors")):
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f"Adapter: {p} -> {size_mb:.2f} MB")
    assert size_mb < 50, f"حجم المحول {p} تجاوز حد GitHub الآمن!"

gh_token = os.environ.get("GITHUB_TOKEN") or getpass.getpass("أدخل GitHub Token (صلاحية repo): ")
subprocess.run(["git", "config", "user.name", "ZeroHunter Swarm10 Trainer"], check=True)
subprocess.run(["git", "config", "user.email", "swarm10@zerohunter.internal"], check=True)
subprocess.run(["git", "add", "-f", "lora_adapters/", "datasets/swarm10/"], check=True)
subprocess.run(["git", "commit", "-m", "Add 10 trained LoRA adapters for Qwen-0.5B Swarm"], check=False)
subprocess.run(["git", "push", f"https://{gh_token}@github.com/sdxdxa56-design/jdjsjsh.git", "HEAD:main"], check=True)
print("✅ تم رفع الـ 10 محولات إلى GitHub بنجاح؛ سيعمل السرب الآن مجاناً على GitHub Actions!")
